In [1]:
# Cell 1: Install all required dependencies
# langchain stack for orchestration, pymupdf for PDF parsing,
# faiss-cpu for local vector search, sentence-transformers for free local embeddings,
# langchain-groq for the cloud LLM, pydantic for output schema, ipywidgets for the GUI.
%pip install langchain langchain-community langchain-openai pymupdf faiss-cpu pydantic python-dotenv
%pip install langchain-groq
%pip install sentence-transformers langchain-huggingface
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2: Imports and configuration
# All imports are collected in one place so dependencies are visible at a glance.
# PDF_PATH is the only config constant — change it here to switch to a different manual.
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List, Optional

PDF_PATH = "sample-service-manual.pdf"

C:\Users\diwak\AppData\Local\Temp\ipykernel_43680\3263454208.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [3]:
# Cell 3: Load and inspect the PDF
# PyMuPDF (fitz) is used because it preserves line-by-line table structure
# better than alternatives for this type of technical manual.
# The preview helps verify that table rows are being read correctly before
# building the pipeline — garbled text here means bad retrieval later.
print(f"Loading PDF: {PDF_PATH}...")
loader = PyMuPDFLoader(PDF_PATH)
documents = loader.load()

print(f"Loaded {len(documents)} pages.")
print("--- Preview of Page 2 (index 1) ---")
print(documents[1].page_content[:1000])

Loading PDF: sample-service-manual.pdf...
Loaded 852 pages.
--- Preview of Page 2 (index 1) ---
Symptom Chart — Suspension System 
Condition 
Possible Sources 
Action 
z Incorrect thrust 
angle (dogtracking) 
z Rear 
suspension 
components 
z INSPECT the rear suspension 
system. CHECK the rear alignment 
for the correct thrust angle. 
REPAIR or INSTALL new 
suspension components as 
necessary. REFER to Section 204-
02 . 
z Vehicle drifts/pulls 
z Unevenly loaded 
or overloaded 
vehicle 
z Tires/tire 
pressure 
z Alignment is not 
within 
specification 
z Brake drag 
z Steering 
components 
z GO to Pinpoint Test A . 
z Front bottoming or 
riding low 
z Worn, damaged 
or incorrect 
springs 
z MEASURE the ride height. REFER 
to Ride Height Measurement in this 
section. INSTALL new springs as 
necessary. Refer to the appropriate 
section in Group 204 for the 
procedure. 
z Worn front 
shock absorbers 
z INSTALL new shock absorbers as 
necessary. Refer to the appropriate 
section in Group 2

In [4]:
# Cell 4: Chunk the document
# chunk_size=2000 is intentionally large to keep full specification tables
# within a single chunk — a 500-char default would split a 20-row table in half.
# chunk_overlap=300 ensures specs at chunk boundaries are not lost.
# RecursiveCharacterTextSplitter splits on natural breaks (paragraphs, then lines)
# rather than cutting blindly at a character count.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=300,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"Document split into {len(chunks)} chunks.")

Document split into 966 chunks.


In [5]:
# Cell 5: Create or load the FAISS vector store
# Embeddings: 'all-MiniLM-L6-v2' is a free, locally-run sentence similarity
# model (~80MB). It requires no API key and is accurate enough for retrieval
# over technical text.
#
# The index is expensive to build (embedding every chunk takes ~30-60s).
# So we check if 'faiss_db_index/' already exists on disk:
#   - If yes  → load it directly, skip all computation.
#   - If no   → build from scratch and save for future runs.
from langchain_huggingface import HuggingFaceEmbeddings

FAISS_INDEX_PATH = "faiss_db_index"

print("Loading local embedding model (first run downloads ~80MB)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

if os.path.exists(FAISS_INDEX_PATH):
    # Index already on disk — load it, no recomputation needed
    vector_store = FAISS.load_local(FAISS_INDEX_PATH, embeddings, allow_dangerous_deserialization=True)
    print(f"Loaded existing vector store from '{FAISS_INDEX_PATH}/'.")
else:
    # First run — build the index and persist it
    print("Building FAISS vector store (this only happens once)...")
    vector_store = FAISS.from_documents(chunks, embeddings)
    vector_store.save_local(FAISS_INDEX_PATH)
    print(f"Vector store built and saved to '{FAISS_INDEX_PATH}/'.")

Loading local embedding model (first run downloads ~80MB)...
Loaded existing vector store from 'faiss_db_index/'.


In [6]:
# Cell 6: Validate retrieval (debug step)
# This cell is intentionally separate from the LLM step.
# Validating retrieval in isolation confirms the vector store is surfacing
# the correct pages before spending API tokens on generation.
# k=10 is used here (vs k=3 in production) to get a broad view of what the index returns.
test_query = "What is the correct torque specification for the front tie-rod end jam nut"
results = vector_store.similarity_search(test_query, k=10)

print(f"Top result for: '{test_query}'")
print(results[0].page_content)
print("\n--- All retrieved chunks ---")
for i, doc in enumerate(results):
    print(f"\n[Chunk {i+1}]")
    print(doc.page_content)
    print("-" * 60)

Top result for: 'What is the correct torque specification for the front tie-rod end jam nut'
Torque Specifications 
a Refer to the procedure in this section 
SECTION 204-02: Rear Suspension 
2014 F-150 Workshop Manual 
SPECIFICATIONS 
Procedure revision date: 10/25/2013 
Description 
Nm lb-ft lb-in 
Shock absorber nuts 
90 
66 
—
Shock absorber shield bolts (SVT Raptor) 
4 
—
35 
Spring shackle-to-frame nut 
185 136 
—
Spring-to-frame nut 
350 258 
—
Spring-to-shackle nut 
185 136 
—
Spring U-bolt nuts a 
—
—
—
Jounce bumper-to-frame bolt 
35 
26 
—
Page 1 sur 1
2014 F-150 Workshop Manual
2014-03-01
file:///C:/TSO/tsocache/VDTOM2_10764/SE2~us~en~file=SE242001.HTM~gen~ref.HT...

--- All retrieved chunks ---

[Chunk 1]
Torque Specifications 
a Refer to the procedure in this section 
SECTION 204-02: Rear Suspension 
2014 F-150 Workshop Manual 
SPECIFICATIONS 
Procedure revision date: 10/25/2013 
Description 
Nm lb-ft lb-in 
Shock absorber nuts 
90 
66 
—
Shock absorber shield bolts (SVT R

In [7]:
# Cell 7: Output schema
# Pydantic models define the shape of every extracted specification.
# VehicleSpec represents a single row: component, type, value, and optional unit.
# SpecList wraps a list of those rows — this mirrors the JSON the LLM produces.
# These models are used in Cell 8 to validate each parsed response.

class VehicleSpec(BaseModel):
    """A single extracted vehicle specification (one row from a spec table)."""
    component: str = Field(..., description="The part or component name (e.g., 'Brake Caliper Bolt').")
    spec_type: str = Field(..., description="The type of specification (e.g., 'Torque', 'Capacity', 'Clearance').")
    value: str = Field(..., description="The numerical value only, no units (e.g., '17').")
    unit: Optional[str] = Field(None, description="The unit of measurement (e.g., 'Nm', 'lb-ft', 'L'). None if not applicable.")

class SpecList(BaseModel):
    """The full extraction result: a list of VehicleSpec rows."""
    specs: List[VehicleSpec]

In [8]:
# Cell 8: LLM setup and JSON extraction helper
#
# LLM choice — Groq (llama-3.3-70b-versatile):
#   A free cloud API backed by Llama 3.3 70B. Originally the pipeline used
#   Ollama for a fully local setup, but that requires significant hardware.
#   Groq provides the same model via a fast, free-tier API.
#   temperature=0 makes the model deterministic — extraction is not creative.
#
# extract_json_from_text:
#   LLMs occasionally wrap JSON in conversational text ("Here is the data:").
#   This helper strips markdown fences, then uses regex to isolate the JSON
#   object regardless of any surrounding filler. Returns a validated SpecList
#   if parsing succeeds, or None on failure.

import json, time, re
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file")

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

def extract_json_from_text(text: str) -> Optional[SpecList]:
    """
    Extracts and validates a JSON object from raw LLM output.
    Strips markdown code fences, isolates the JSON block via regex,
    parses it, and validates it against the SpecList schema.
    Returns a SpecList on success, or None if parsing/validation fails.
    """
    try:
        text = text.replace("```json", "").replace("```", "")
        match = re.search(r"\{.*\}", text, re.DOTALL)
        raw = json.loads(match.group(0)) if match else json.loads(text)
        return SpecList(**raw)  # validate against Pydantic schema
    except Exception:
        return None

In [9]:
# Cell 9: Batch extraction and save results
#
# Prompt design:
#   The UNIT PATTERN RULE is the key domain insight. Service manuals list
#   torque in three columns: Nm -> lb-ft -> lb-in. A dash means that column
#   is skipped. Without this rule, the model misassigns values to units.
#   The SEPARATION RULE enforces clean output: value is a number, unit is a code.
#
# k=3 retrieval:
#   Only the 3 most relevant chunks are sent to the LLM per query.
#   This keeps the prompt within Groq's token limit while staying focused.
#   Tradeoff: specs spread across more than 3 chunks may be partially missed.

EXTRACTION_PROMPT = """
You are a highly accurate technical data extractor.
Your task is to analyze the provided text and extract ALL specifications related to the user's query.

CRITICAL INSTRUCTIONS:
1. The text may contain long tables (20+ rows). You MUST extract EVERY row.
2. UNIT PATTERN RULE:
   - Service manuals list torque in this order: Nm (Metric) -> lb-ft (Imperial) -> lb-in (Small Imperial).
   - If you see "12 — 106", the first number (12) is Nm, the dash means lb-ft is absent, and 106 is lb-in.
3. SEPARATION RULE:
   - "value" must contain ONLY the number (e.g., "17"). No text, no units.
   - "unit" must contain ONLY the unit code (e.g., "Nm").

Output Format:
{{
    "specs": [
        {{ "component": "part name", "spec_type": "Torque", "value": "17", "unit": "Nm" }}
    ]
}}

If no relevant data is found, return: {{ "specs": [] }}

Context:
{context}

Query: {question}
"""

QUERIES = [
    "Torque specifications for front suspension",
    "Torque specifications for braking system",
    "Fluid capacities"
]

all_extracted_data = []
prompt = ChatPromptTemplate.from_template(EXTRACTION_PROMPT)
chain = prompt | llm

print("Starting batch extraction...")

for query in QUERIES:
    print(f"  Processing: {query}...")
    start_ts = time.time()

    # Retrieve the 3 most relevant chunks for this query
    docs = vector_store.similarity_search(query, k=3)
    context_text = "\n\n".join([d.page_content for d in docs])

    try:
        response = chain.invoke({"context": context_text, "question": query})
        result = extract_json_from_text(response.content)

        if result and result.specs:
            # Convert validated Pydantic objects back to plain dicts for JSON serialization
            items = [s.model_dump() for s in result.specs]
            all_extracted_data.extend(items)
            print(f"  Found {len(items)} specs in {time.time() - start_ts:.2f}s.")
        elif result:
            print("  Valid response, but no specs found for this query.")
        else:
            print("  Could not parse a valid JSON response from the model.")

    except Exception as e:
        print(f"  API error: {e}")

# Save all extracted specs to disk
OUTPUT_FILE = "vehicle_specs.json"
with open(OUTPUT_FILE, "w") as f:
    json.dump(all_extracted_data, f, indent=4)

print(f"\nDone. Saved {len(all_extracted_data)} total specs to '{OUTPUT_FILE}'.")
print("\nFirst 5 results:")
print(json.dumps(all_extracted_data[:5], indent=2))

Starting batch extraction...
  Processing: Torque specifications for front suspension...
  Found 30 specs in 3.01s.
  Processing: Torque specifications for braking system...
  Found 43 specs in 3.06s.
  Processing: Fluid capacities...
  Found 2 specs in 0.60s.

Done. Saved 75 total specs to 'vehicle_specs.json'.

First 5 results:
[
  {
    "component": "Brake disc shield bolts",
    "spec_type": "Torque",
    "value": "17",
    "unit": "Nm"
  },
  {
    "component": "Brake disc shield bolts",
    "spec_type": "Torque",
    "value": "150",
    "unit": "lb-in"
  },
  {
    "component": "Brake hose bracket bolt",
    "spec_type": "Torque",
    "value": "12",
    "unit": "Nm"
  },
  {
    "component": "Brake hose bracket bolt",
    "spec_type": "Torque",
    "value": "106",
    "unit": "lb-in"
  },
  {
    "component": "Lower arm forward and rearward nuts",
    "spec_type": "Torque",
    "value": "350",
    "unit": "Nm"
  }
]


In [18]:
# Cell 10: Interactive GUI — Predii AI Service Hub
#
# Built with ipywidgets so the entire interface runs inside the notebook.
# The user can either pick a preset query from the dropdown or type a custom one.
# Selecting a preset auto-fills the text input, which the user can also edit.
#
# on_search_click runs the full RAG pipeline live on each button press:
#   1. Retrieve top-3 chunks from the FAISS index
#   2. Send them to Groq Llama 3.3 with the extraction prompt
#   3. Parse and validate the response with extract_json_from_text
#   4. Render results as a styled HTML table, or show an error message
#
# clean_and_parse_json is the GUI-side parser — same logic as extract_json_from_text
# but returns an empty dict on failure instead of None, keeping the UI stable.

import ipywidgets as widgets
from IPython.display import display, HTML

# GUI uses its own LLM instance so it is self-contained and can run independently
gui_llm = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY
)

GUI_PROMPT = """
You are a technical assistant. Extract ALL specifications for: '{question}'.

CRITICAL INSTRUCTIONS:
1. Torque tables follow the pattern: Nm -> lb-ft -> lb-in. A dash means that column is absent.
2. For fluids or part numbers with no unit, return null for the unit field.
3. Extract EVERY row. For torque values, the value field must contain numbers only.

Output JSON: {{ "specs": [ {{ "component": "...", "spec_type": "...", "value": "...", "unit": "..." }} ] }}
If no data found, return {{ "specs": [] }}

Context:
{context}
"""

def clean_and_parse_json(raw_response: str) -> dict:
    """
    GUI-side JSON parser. Same regex approach as extract_json_from_text but
    returns {"specs": []} instead of None on failure to keep the UI stable.
    """
    try:
        text = str(raw_response).replace("```json", "").replace("```", "")
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        return {"specs": []}
    except Exception:
        return {"specs": []}

# --- UI Layout ---
header_html = """
<style>
    .mechanic-header {
        background: linear-gradient(135deg, #1F2A38, #34495E);
        color: white;
        padding: 16px 22px;
        border-radius: 10px 10px 0 0;
        font-family: sans-serif;
        border-left: 4px solid #E67E22;
    }
    .mechanic-header h2 { margin: 0; font-size: 17px; font-weight: 600; letter-spacing: 0.3px; }
    .mechanic-subheader { font-size: 12px; opacity: 0.75; margin-top: 4px; }
    .result-table {
        width: 100%; border-collapse: collapse; margin-top: 15px;
        font-family: sans-serif; font-size: 14px; text-align: left;
    }
    .result-table th {
        background: #ECF0F1; color: #2C3E50; padding: 10px;
        border-bottom: 2px solid #E67E22;
    }
    .result-table td {
        border-bottom: 1px solid #eee; padding: 8px; color: #333; word-wrap: break-word;
    }
    .result-table tr:hover td { background: #FAFAFA; }
    .highlight-val { color: #E67E22; font-weight: bold; }
</style>
<div class="mechanic-header">
    <h2>Predii AI: Service Hub</h2>
    <div class="mechanic-subheader">Powered by Groq Llama 3.3</div>
</div>
"""

PRESET_QUERIES = [
    "--- Select a Quick Query ---",
    "Torque specifications for front suspension",
    "Torque specifications for braking system",
    "Service materials",
    "Wheel alignment specifications",
    "Grease and lubricants"
]

header_widget = widgets.HTML(value=header_html)
dropdown = widgets.Dropdown(options=PRESET_QUERIES, layout=widgets.Layout(width='98%'))
query_input = widgets.Text(placeholder='...or type a specific question here', layout=widgets.Layout(width='98%'))
search_btn = widgets.Button(description=' Extract Data', icon='search', button_style='primary', layout=widgets.Layout(width='200px', height='40px'))
output_area = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '15px', 'margin_top': '15px', 'min_height': '100px'})

def on_dropdown_change(change):
    """Auto-fill the text input when a preset is selected."""
    if change['new'] != "--- Select a Quick Query ---":
        query_input.value = change['new']

def on_search_click(b):
    """Run the full RAG pipeline and render results on button click."""
    output_area.clear_output()
    q = query_input.value

    if not q or q == "--- Select a Quick Query ---":
        with output_area:
            display(HTML("<b style='color:orange;'>Please select or type a query.</b>"))
        return

    with output_area:
        display(HTML("<b>Querying... please wait...</b>"))
        start_ts = time.time()

        try:
            # Step 1: Retrieve the most relevant chunks
            docs = vector_store.similarity_search(q, k=3)
            context = "\n\n".join([d.page_content for d in docs])

            # Step 2: Generate with the LLM
            gui_prompt = ChatPromptTemplate.from_template(GUI_PROMPT)
            response = (gui_prompt | gui_llm).invoke({"context": context, "question": q})

            # Step 3: Parse the JSON response
            data = clean_and_parse_json(response.content)
            specs = data.get("specs", [])
            elapsed = time.time() - start_ts
            output_area.clear_output()

            if specs:
                print(f"Found {len(specs)} specs in {elapsed:.2f}s:")
                rows = "".join(
                    f"<tr><td>{s.get('component','-')}</td><td>{s.get('spec_type','-')}</td>"
                    f"<td class='highlight-val'>{s.get('value','-')}</td><td>{s.get('unit','') or ''}</td></tr>"
                    for s in specs
                )
                table = f"<table class='result-table'><thead><tr><th>Component</th><th>Type</th><th>Value</th><th>Unit</th></tr></thead><tbody>{rows}</tbody></table>"
                display(HTML(table))
            else:
                display(HTML(f"<div style='color:red; padding:10px;'>No data found for: '{q}'</div>"))

        except Exception as e:
            output_area.clear_output()
            print(f"Error: {e}")

dropdown.observe(on_dropdown_change, names='value')
search_btn.on_click(on_search_click)

controls = widgets.VBox([
    widgets.Label(value="Query Type Select:"), dropdown,
    widgets.Label(value="Custom Query:"), query_input,
    widgets.Box([search_btn], layout=widgets.Layout(margin='15px 0 0 0'))
], layout=widgets.Layout(padding='20px'))

display(widgets.VBox([header_widget, controls, output_area]))